In [1]:
from __future__ import division, print_function
import sys, os, glob, time, warnings, gc
import numpy as np
# import matplotlib
# matplotlib.use("Agg")
import matplotlib.pyplot as plt
from astropy.table import Table, vstack, hstack, join
import fitsio
# from astropy.io import fits

In [2]:
params = {'legend.fontsize': 'large',
         'axes.labelsize': 'large',
         'axes.titlesize':'large',
         'xtick.labelsize':'large',
         'ytick.labelsize':'large',
         'figure.facecolor':'w'} 
plt.rcParams.update(params)

In [3]:
cat = Table(fitsio.read('/Users/rongpu/Documents/Data/desi_data/everest/main_cumulative_lrg.fits'))
cat['EFFTIME_ELG'] = 8.60 * cat['TSNR2_ELG']
cat['EFFTIME_LRG'] = 12.15 * cat['TSNR2_LRG']

# Remove FIBERSTATUS!=0 fibers
mask = cat['COADD_FIBERSTATUS']==0
print('FIBERSTATUS',np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
cat = cat[mask]

# Remove "no data" fibers
mask = cat['ZWARN'] & 2**9==0
print('No data', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
cat = cat[mask]

# Apply LRG mask
mask = cat['lrg_mask']==0
print('LRG mask', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
cat = cat[mask]

# # Remove QSO targets
# mask = cat['DESI_TARGET'] & 2**2 ==0
# print('Remove QSO targets', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
# cat = cat[mask]

# Require a minimum depth
min_depth = 800.
mask = cat['EFFTIME_LRG']>min_depth
print('Min depth', np.sum(mask), np.sum(~mask), np.sum(mask)/len(mask))
cat = cat[mask]

# Julien's bad fibers list + my list of worst-performing fibers; bad_fibers-everest.ipynb
# bad_fibers = np.loadtxt('/global/cfs/cdirs/desi/users/rongpu/spectro/everest/misc/bad_fibers_20211117.txt', dtype=int)
bad_fibers = np.loadtxt('/Users/rongpu/Documents/Data/desi_data/everest/misc/bad_fibers_20211117.txt', dtype=int)
print(len(bad_fibers))
mask_bad = np.in1d(cat['FIBER'], bad_fibers)
print('Bad fibers', np.sum(~mask_bad), np.sum(mask_bad), np.sum(mask_bad)/len(mask_bad))
cat = cat[~mask_bad]
print(len(cat), len(np.unique(cat['TARGETID'])))

FIBERSTATUS 340944 5119 0.014792104327824702
No data 340944 0 0.0
LRG mask 306665 34279 0.10054143789009339
Min depth 299348 7317 0.976140087717868
333
Bad fibers 278644 20704 0.06916364899715381
278644 278616


In [5]:
# Custom DELTACHI2 vs z cut
d = (10**(3 - 3.5*cat['Z']))
mask_remove = (d>30) & (cat['DELTACHI2']<30)
mask_remove |= (d<30) & (cat['DELTACHI2']<d)
mask_remove |= (cat['DELTACHI2']<10)
mask_quality = cat['ZWARN']==0
mask_quality &= cat['Z']<1.4
mask_quality &= (~mask_remove)

print(np.sum(~mask_quality)/len(mask_quality))
cat = cat[mask_quality]
print(len(cat))

0.019813812606767057
273123


In [6]:
mask_star = (cat['SPECTYPE']=='STAR') | (cat['Z']<0.0003)
cat['SPECTYPE'][mask_star] = 'STAR'
print(np.sum(mask_star)/len(mask_star))

0.004584015260523647


In [10]:
t = Table()
t['SPECTYPE'], t['count'] = np.unique(cat['SPECTYPE'], return_counts=True)
t['frac (%)'] = t['count']/len(cat)*100
t['frac (%)'].format = '%.2f'
t.sort('count')
t

SPECTYPE,count,frac (%)
str6,int64,float64
STAR,1252,0.46
QSO,5662,2.07
GALAXY,266209,97.47


In [11]:
# Fraction of QSO targets
mask = cat['DESI_TARGET'] & 2**2>0
print(np.sum(mask)/len(mask))

0.013459137458214797


In [12]:
# Exclude QSO targets
mask = cat['DESI_TARGET'] & 2**2==0
print(np.sum(mask), np.sum(mask)/len(mask))

t = Table()
t['SPECTYPE'], t['count'] = np.unique(cat['SPECTYPE'][mask], return_counts=True)
t['frac (%)'] = t['count']/np.sum(mask)*100
t['frac (%)'].format = '%.2f'
t.sort('count')
t

269447 0.9865408625417852


SPECTYPE,count,frac (%)
str6,int64,float64
STAR,1178,0.44
QSO,3231,1.20
GALAXY,265038,98.36


In [13]:
# Select QSO targets
mask = cat['DESI_TARGET'] & 2**2>0
print(np.sum(mask), np.sum(mask)/len(mask))

t = Table()
t['SPECTYPE'], t['count'] = np.unique(cat['SPECTYPE'][mask], return_counts=True)
t['frac (%)'] = t['count']/np.sum(mask)*100
t['frac (%)'].format = '%.2f'
t.sort('count')
t

3676 0.013459137458214797


SPECTYPE,count,frac (%)
str6,int64,float64
STAR,74,2.01
GALAXY,1171,31.86
QSO,2431,66.13


In [14]:
# Downweight QSO targets and recalculate the fractions
mask_qso = cat['DESI_TARGET'] & 2**2>0
print(np.sum(mask_qso)/len(mask_qso))
cat['weight'] = 1.
cat['weight'][mask_qso] = 0.5

mask = cat['SPECTYPE']=='STAR'
print('Star fraction: {:.2f}%'.format(100*np.sum(mask*cat['weight'])/np.sum(cat['weight'])))
mask = cat['SPECTYPE']=='QSO'
print('QSO fraction: {:.2f}%'.format(100*np.sum(mask*cat['weight'])/np.sum(cat['weight'])))

0.013459137458214797
Star fraction: 0.45%
QSO fraction: 1.64%


------
# Spectype of masked LRG targets

In [15]:
cat = Table(fitsio.read('/Users/rongpu/Documents/Data/desi_data/everest/main_cumulative_lrg.fits'))
cat['EFFTIME_ELG'] = 8.60 * cat['TSNR2_ELG']
cat['EFFTIME_LRG'] = 12.15 * cat['TSNR2_LRG']

# Remove FIBERSTATUS!=0 fibers
mask = cat['COADD_FIBERSTATUS']==0
print('FIBERSTATUS',np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
cat = cat[mask]

# Remove "no data" fibers
mask = cat['ZWARN'] & 2**9==0
print('No data', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
cat = cat[mask]

# Apply LRG mask
mask = cat['lrg_mask']!=0
print('LRG mask', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
cat = cat[mask]

# # Remove QSO targets
# mask = cat['DESI_TARGET'] & 2**2 ==0
# print('Remove QSO targets', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
# cat = cat[mask]

# Require a minimum depth
min_depth = 800.
mask = cat['EFFTIME_LRG']>min_depth
print('Min depth', np.sum(mask), np.sum(~mask), np.sum(mask)/len(mask))
cat = cat[mask]

# Julien's bad fibers list + my list of worst-performing fibers; bad_fibers-everest.ipynb
# bad_fibers = np.loadtxt('/global/cfs/cdirs/desi/users/rongpu/spectro/everest/misc/bad_fibers_20211117.txt', dtype=int)
bad_fibers = np.loadtxt('/Users/rongpu/Documents/Data/desi_data/everest/misc/bad_fibers_20211117.txt', dtype=int)
print(len(bad_fibers))
mask_bad = np.in1d(cat['FIBER'], bad_fibers)
print('Bad fibers', np.sum(~mask_bad), np.sum(mask_bad), np.sum(mask_bad)/len(mask_bad))
cat = cat[~mask_bad]
print(len(cat), len(np.unique(cat['TARGETID'])))

FIBERSTATUS 340944 5119 0.014792104327824702
No data 340944 0 0.0
LRG mask 34279 306665 0.8994585621099066
Min depth 33428 851 0.9751743049680562
333
Bad fibers 31135 2293 0.06859518966136173
31135 31133


In [16]:
# Custom DELTACHI2 vs z cut
d = (10**(3 - 3.5*cat['Z']))
mask_remove = (d>30) & (cat['DELTACHI2']<30)
mask_remove |= (d<30) & (cat['DELTACHI2']<d)
mask_remove |= (cat['DELTACHI2']<10)
mask_quality = cat['ZWARN']==0
mask_quality &= cat['Z']<1.4
mask_quality &= (~mask_remove)

print(np.sum(~mask_quality)/len(mask_quality))
cat = cat[mask_quality]
print(len(cat))

0.03170065842299663
30148


In [17]:
mask_star = (cat['SPECTYPE']=='STAR') | (cat['Z']<0.0003)
cat['SPECTYPE'][mask_star] = 'STAR'
print(np.sum(mask_star)/len(mask_star))

0.10471673079474592


In [18]:
t = Table()
t['SPECTYPE'], t['count'] = np.unique(cat['SPECTYPE'], return_counts=True)
t['frac (%)'] = t['count']/len(cat)*100
t['frac (%)'].format = '%.1f'
t.sort('count')
t

SPECTYPE,count,frac (%)
str6,int64,float64
QSO,627,2.1
STAR,3157,10.5
GALAXY,26364,87.4


In [19]:
# Fraction of QSO targets
mask = cat['DESI_TARGET'] & 2**2>0
print(np.sum(mask)/len(mask))

0.020134005572508957


In [20]:
# Exclude QSO targets
mask = cat['DESI_TARGET'] & 2**2==0
print(np.sum(mask), np.sum(mask)/len(mask))

t = Table()
t['SPECTYPE'], t['count'] = np.unique(cat['SPECTYPE'][mask], return_counts=True)
t['frac (%)'] = t['count']/np.sum(mask)*100
t['frac (%)'].format = '%.1f'
t.sort('count')
t

29541 0.979865994427491


SPECTYPE,count,frac (%)
str6,int64,float64
QSO,341,1.2
STAR,2994,10.1
GALAXY,26206,88.7


In [21]:
# Select QSO targets
mask = cat['DESI_TARGET'] & 2**2>0
print(np.sum(mask), np.sum(mask)/len(mask))

t = Table()
t['SPECTYPE'], t['count'] = np.unique(cat['SPECTYPE'][mask], return_counts=True)
t['frac (%)'] = t['count']/np.sum(mask)*100
t['frac (%)'].format = '%.1f'
t.sort('count')
t

607 0.020134005572508957


SPECTYPE,count,frac (%)
str6,int64,float64
GALAXY,158,26.0
STAR,163,26.9
QSO,286,47.1


In [22]:
# Downweight QSO targets and recalculate the fractions
mask_qso = cat['DESI_TARGET'] & 2**2>0
print(np.sum(mask_qso)/len(mask_qso))
cat['weight'] = 1.
cat['weight'][mask_qso] = 0.3

mask = cat['SPECTYPE']=='STAR'
print('Star fraction: {:.2f}%'.format(100*np.sum(mask*cat['weight'])/np.sum(cat['weight'])))
mask = cat['SPECTYPE']=='QSO'
print('QSO fraction: {:.2f}%'.format(100*np.sum(mask*cat['weight'])/np.sum(cat['weight'])))

0.020134005572508957
Star fraction: 10.24%
QSO fraction: 1.44%
